# Double Descent study — `credit` dataset

**What is double descent?** Normally we expect: train longer → training loss keeps dropping, but test loss goes down then **up** (classic overfitting), so you stop early. *Deep double descent* (Nakkiran et al., 2019) says that if you keep training **far past** the point where training loss hits ~zero, test loss can come **down a second time** — sometimes below its first minimum.

```
test loss:   \____/‾‾‾\____      ← down, UP (the bump), down again
train loss:  \_____________      ← hits ~0 somewhere in the middle
```

**The recipe used here (your settings):**

| setting | value | why |
|---|---|---|
| optimizer | **vanilla SGD** (momentum 0) | slow, plain updates — the classic setting |
| learning rate | **1e-5** | very small steps |
| batch size | **16** | small batches = many noisy updates |
| epochs | **4000** | must train far past interpolation to see the second descent |
| dropout / weight-decay / L1 | **0** | regularization *suppresses* double descent |

**Three models, measured every epoch:** `x` (raw only), `x+tree` (raw + RF split bits), `x+tree+deep` (FULL) — recording **train / validation / test** × **loss, AUC, accuracy**.

## 💾 Crash-proof: checkpoints on Google Drive
Every **10 epochs** the model + optimizer + RNG + full history are saved to your Drive, and the **previous checkpoint is deleted** (so only the newest file exists — ep 10, then 20, then 30…).

**If Colab disconnects: just re-run the same cell.** It finds the checkpoint and continues from the exact epoch it died on, with the loss curve joining seamlessly. Nothing is lost but the last <10 epochs.

> ⚠️ **Runtime ≈ 1 hour per model, ~3 hours total.** Each model has its own cell.
> Test metrics are recorded **only to plot the curve** — model selection never uses them.

In [ ]:
# 1 · GPU check
import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU -> Runtime > Change runtime type > GPU')

In [ ]:
# 2 · mount Google Drive (for the checkpoints)
from google.colab import drive
drive.mount('/content/drive')
import os
CKPT = '/content/drive/MyDrive/tkce_double_descent'
os.makedirs(CKPT, exist_ok=True)
print('checkpoints ->', CKPT)
print('already there:', os.listdir(CKPT) or '(empty — fresh start)')

In [ ]:
# 3 · get the code
%cd /content
!git clone https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull

In [ ]:
# 4 · install deps
!pip install -q openml catboost optuna

In [ ]:
# 5 · the shared configuration (your exact settings)
CFG = ('--task 361055 '                      # credit
       '--optimizer sgd --momentum 0 '       # vanilla SGD
       '--lr 1e-5 '
       '--batch-size 16 '
       '--epochs 4000 '
       '--dropout 0 --weight-decay 0 --l1 0 '  # no regularization
       '--eval-every 1 '                       # metrics EVERY epoch
       '--log-every 100 --flush-every 100 '
       f'--ckpt-dir {CKPT} --ckpt-every 10 '   # <- Drive checkpoints
       '--device auto')
print(CFG)

## Run the three models
≈1 hour each. **If a cell dies, just run it again** — it resumes from the last Drive checkpoint.

*(To force a clean restart of a model, add `--fresh` to that cell.)*

In [ ]:
# 6 · MODEL 1 of 3 — raw features only (the baseline)
!python -u run_double_descent.py {CFG} --views 'x'

In [ ]:
# 7 · MODEL 2 of 3 — raw + tree split-bits
!python -u run_double_descent.py {CFG} --views 'x+tree'

In [ ]:
# 8 · MODEL 3 of 3 — FULL (raw + tree + deep extractor)
!python -u run_double_descent.py {CFG} --views 'x+tree+deep'

## Combine and analyse

In [ ]:
# 9 · merge the three histories + build the comparison figure
import os, json, pandas as pd
from run_double_descent import compare_figure
D = 'results/double_descent'
paths = {'x': f'{D}/dd_credit_x.csv',
         'x+tree': f'{D}/dd_credit_x-tree.csv',
         'x+tree+deep': f'{D}/dd_credit_x-tree-deep.csv'}
# fall back to the Drive partial CSVs for any model still running / interrupted
for k, p in list(paths.items()):
    if not os.path.exists(p):
        alt = f'{CKPT}/dd_credit_{k}_partial.csv'
        if os.path.exists(alt):
            paths[k] = alt; print(f'using Drive partial for {k}')
have = {k: v for k, v in paths.items() if os.path.exists(v)}
print('found:', list(have))
df = pd.concat([pd.read_csv(v) for v in have.values()], ignore_index=True)
df.to_csv(f'{D}/dd_credit_epochs.csv', index=False)
ceiling = json.load(open(f'{D}/dd_credit.json'))['tree_ceiling']
compare_figure(df, ceiling, f'{D}/dd_credit_compare.png')
print('rows:', len(df), '| tree ceiling:', round(ceiling, 4))

In [ ]:
# 10 · DOUBLE-DESCENT LANDMARKS  (does test loss dip -> rise -> dip again?)
for m in df.model.unique():
    g = df[df.model == m].sort_values('epoch').reset_index(drop=True)
    first_half = g[g.epoch <= g.epoch.max() / 2]
    iA = first_half.test_loss.idxmin()                 # first dip
    iP = g.loc[iA:].test_loss.idxmax()                 # the bump peak
    iB = g.loc[iP:].test_loss.idxmin()                 # second dip
    rise = g.loc[iP, 'test_loss'] - g.loc[iA, 'test_loss']
    fall = g.loc[iP, 'test_loss'] - g.loc[iB, 'test_loss']
    print(f'\n{m}')
    print(f"   first dip : ep {int(g.loc[iA,'epoch']):5d}  loss={g.loc[iA,'test_loss']:.4f}  auc={g.loc[iA,'test_auc']:.4f}")
    print(f"   bump peak : ep {int(g.loc[iP,'epoch']):5d}  loss={g.loc[iP,'test_loss']:.4f}")
    print(f"   later dip : ep {int(g.loc[iB,'epoch']):5d}  loss={g.loc[iB,'test_loss']:.4f}  auc={g.loc[iB,'test_auc']:.4f}")
    print(f'   rise after first dip = {rise:+.4f} | fall after peak = {fall:+.4f}')
    print(f"   final train_loss = {g.train_loss.iloc[-1]:.5f}  (near 0 = interpolated)")
    print('   ->', 'LOOKS LIKE DOUBLE DESCENT' if rise > 0.005 and fall > 0.005
          else 'no clear second descent (single descent / plain overfitting)')

In [ ]:
# 11 · show every figure
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/double_descent/*.png')):
    print('==', f, '==')
    display(Image(f))

In [ ]:
# 12 · download everything (also copy results to Drive for safekeeping)
import shutil
from google.colab import files
shutil.copytree('results/double_descent', f'{CKPT}/results', dirs_exist_ok=True)
shutil.make_archive('double_descent_credit', 'zip', 'results/double_descent')
files.download('double_descent_credit.zip')